# **Pavan S - 24BAD085**

---

# **Defining the libraries required to perform tasks**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc
)

# **Loading the Dataset**

In [ ]:
df = pd.read_csv("/data/LICI - Daily data.csv")
df_1 = pd.read_csv("/data/LICI - minute data.csv")
df_2 = pd.read_csv("/data/LICI - 3 minute data.csv")
df_3 = pd.read_csv("/data/LICI - 5 minute data.csv")
df_4 = pd.read_csv("/data/LICI - 10 minute data.csv")

# **Creating Target Variables and input features**

In [ ]:
df_2['Price_Movement'] = np.where(df_2['close'] > df_2['open'], 1, 0)

X = df_2[['open', 'high', 'low', 'volume']]
y = df_2['Price_Movement']

# **Missing data**

In [ ]:
X = X.fillna(X.mean())

# **Feature Scaling**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# **Train Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# **Logistic Regression Model**

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# **Predictions**

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# **Evaluation Metrics**

In [ ]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))


# **Confusion Matrix**

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# **ROC Curve**

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# **Feature Importance**

In [ ]:
importance = pd.Series(
    model.coef_[0],
    index=['Open', 'High', 'Low', 'Volume']
)

importance.sort_values().plot(kind='barh')
plt.title("Feature Importance")
plt.show()

# **Hyper Parameter and Regularization**

In [16]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='f1'
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Best Parameters:", grid.best_params_)
print("Tuned Accuracy :", accuracy_score(y_test, y_pred_best))
print("Tuned F1 Score :", f1_score(y_test, y_pred_best))

Best Parameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Tuned Accuracy : 0.820919175911252
Tuned F1 Score : 0.7926605504587156
Best Parameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Tuned Accuracy : 0.820919175911252
Tuned F1 Score : 0.7926605504587156
